In [ ]:
import argparse
import os
import time
import torch
import pandas as pd
from src import load_finetuned_model_lens_from_dir
from src.utils import (
    filter_correct_data,
    create_full_AOS_dataset,
    create_aos_sequence_variant,
    build_eap_dataset
)
import src
import src.utils
import importlib
import json
import numpy as np
import os
import json
from dotenv import load_dotenv
load_dotenv()

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [ ]:
model_path = 'outputs/models/eap/circuit-eng_finetune-eng/seed_123/aos_sequence_variants/2025-06-21 11:05:38.594090_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20'
dataset_path = 'hotel_dataset/counterfacts/tflens_hotel-en_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
filtered_data_path = 'test/eng_debug_filtered.csv'
full_aos_path = 'test/eng_debug_full_aos.csv'
sequence_variants_path = 'test/eng_debug_sequence_variant.csv'
eap_output_path = 'test/eng_debug_eap_dataset.csv'

## Filter Dataset

### Normal Pipeline

In [ ]:
print("Starting create EAP dataset pipeline...")

# === Load Model ===
print("Loading fine-tuned model...")
model = load_finetuned_model_lens_from_dir(model_path)
device = (
	torch.device("mps") if torch.backends.mps.is_available()
	else torch.device("cuda") if torch.cuda.is_available()
	else torch.device("cpu")
)
model.to(device)
model.eval()


In [ ]:
# === Step 1: Filter Correct Predictions ===
print("Reading dataset and filtering correct predictions...")
df = pd.read_csv(dataset_path)
print(f"Dataset loaded from {dataset_path} ({len(df)} rows)")
os.makedirs(os.path.dirname(filtered_data_path), exist_ok=True)
filtered_df = filter_correct_data(
	model,
	df,
	"original_sentence",
	"original_triplet",
	filter_mode="AOS",
	filter_only_correct=False,
	save_path=filtered_data_path
)
filtered_df[filtered_df['is_match']].to_csv(filtered_data_path, index=False)
print(f"Filtered data saved to {filtered_data_path} ({len(filtered_df)} rows)")

In [ ]:
debug_view = filtered_df[~filtered_df['is_match']].copy()

### Check all models

In [ ]:
# List all files in a directory recursively, but stop at the last folder before a file
import os
def list_files_recursively(directory):
	file_list = []
	for root, dirs, files in os.walk(directory):
		for file in files:
			file_list.append(os.path.join(root, file))
	return file_list
files = list_files_recursively('outputs/models/eap')
models = [os.path.dirname(f) for f in files]
models = list(set(models))
models

In [ ]:
temp_path = 'hotel_dataset/counterfacts/tflens_hotel-en_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
df_temp = pd.read_csv(temp_path)
df_temp['original_sentence'] = df_temp['original_sentence'].apply(lambda x: f" {x}")
df_temp.to_csv(temp_path.replace('.csv', '_spaceprefix.csv'), index=False)

In [ ]:
filtered_dfs = {}
for model_path in models:
	model = load_finetuned_model_lens_from_dir(model_path)
	device = (
		torch.device("mps") if torch.backends.mps.is_available()
		else torch.device("cuda") if torch.cuda.is_available()
		else torch.device("cpu")
	)
	model.to(device)
	model.eval()

	print(model_path)

	if 'indo' in model_path:
		dataset_path = 'hotel_dataset/counterfacts/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
		language = 'indo'
	elif 'eng' in model_path:
		dataset_path = 'hotel_dataset/counterfacts/tflens_hotel-en_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
		language = 'eng'
	elif 'sunda' in model_path:
		dataset_path = 'hotel_dataset/counterfacts/franken_hotel-su_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
		language = 'sunda'
	else:
		raise ValueError("Unknown model language in path: " + model_path)

	seed = model_path.split('/')[4]
	df = pd.read_csv(dataset_path)
	filtered_data_path = f'temp/{language}_{seed}.csv'
	os.makedirs(os.path.dirname(filtered_data_path), exist_ok=True)
	id = filtered_data_path.split('/')[-1]
	filtered_df = filter_correct_data(
		model,
		df,
		"original_sentence",
		"original_triplet",
		filter_mode="AOS",
		filter_only_correct=True,
		save_path=filtered_data_path
	)
	filtered_dfs[id] = filtered_df.copy()

In [ ]:
filtered_dfs.keys()

In [ ]:
indexes = set()
first = True
for df in filtered_dfs.values():
	if first:
		indexes = set(df['index'].tolist())
		first = False
	else:
		# Get the intersection of indexes
		indexes.intersection_update(df['index'].tolist())

In [ ]:
len(indexes)

In [ ]:
# dataset_path = 'hotel_dataset/counterfacts/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
# language = 'indo'
# dataset_path = 'hotel_dataset/counterfacts/tflens_hotel-en_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
# language = 'eng'
dataset_path = 'hotel_dataset/counterfacts/franken_hotel-su_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
language = 'sunda'

df = pd.read_csv(dataset_path)

# Only take df with index same as indexes
filtered_df = df[df['index'].isin(indexes)].copy()
filtered_df.to_csv(dataset_path.replace('.csv', f'_filtered.csv'), index=False)

## Create Full AOS Dataset

In [ ]:
# === Step 2: Create Full AOS Dataset ===
print("Creating full AOS dataset...")
os.makedirs(os.path.dirname(full_aos_path), exist_ok=True)
full_aos_df = create_full_AOS_dataset(filtered_data_path)
full_aos_df.to_csv(full_aos_path, index=False)
print(f"Full AOS dataset saved to {full_aos_path} ({len(full_aos_df)} rows)")

## Create Sequence Variants

In [ ]:
# === Step 3: Create AOS Sequence Variants ===
print("Creating AOS sequence variants...")
os.makedirs(os.path.dirname(sequence_variants_path), exist_ok=True)
sequence_df = create_aos_sequence_variant(full_aos_path)
sequence_df.to_csv(sequence_variants_path, index=False)
print(f"AOS sequence variants saved to {sequence_variants_path} ({len(sequence_df)} rows)")

In [ ]:
sequence_df

## Build EAP Dataset

In [ ]:
import argparse
import os
import time
import torch
import pandas as pd
from src import load_finetuned_model_lens_from_dir
from src.utils import (
    filter_correct_data,
    create_full_AOS_dataset,
    create_aos_sequence_variant,
    build_eap_dataset
)
import src
import src.utils
import importlib

importlib.reload(src)
importlib.reload(src.utils)

In [ ]:
# === Step 4: Build EAP Dataset ===
print("Building EAP dataset...")
os.makedirs(os.path.dirname(eap_output_path), exist_ok=True)
eap_df = src.utils.build_eap_dataset(
	model=model,
	df=sequence_df,
	sentence_col="original_sentence",
	triplet_col="original_triplet",
	corrupted_col="counterfact3_aspect_replaced",
	corrupted_triplet_col="counterfact_triplet3_aspect_replaced",
	filer_same_length_counterfactuals=True,
	suffix="[A] [O] [S]"
)
eap_df.to_csv(eap_output_path, index=False)
print(f"EAP dataset saved to {eap_output_path} ({len(eap_df)} rows)")

## Get Valid Counterfact Data Candidates Between All Full-Finetuned Models

### Get Intersection Between Models

In [ ]:
import re

def parse_aos_triplet(triplet_str):
    """
    Parse AOS triplet string to extract Aspect, Opinion, and Sentiment.
    Works with any ordering of [A], [O], [S] tags and any length of text between them.
    
    Args:
        triplet_str: String like "[A] service [O] very good [S] positive" or
                    "[O] very good [A] service [S] positive" or any other ordering
    
    Returns:
        tuple: (aspect, opinion, sentiment) or None if parsing fails
    """
    # Remove extra whitespaces and strip
    triplet_str = ' '.join(triplet_str.split()).strip()
    
    # Initialize variables
    aspect = None
    opinion = None
    sentiment = None
    
    # Find all tags and their positions
    tags = ['A', 'O', 'S']
    tag_positions = {}
    
    for tag in tags:
        pattern = rf'\[{tag}\]'
        match = re.search(pattern, triplet_str)
        if match:
            tag_positions[tag] = match.start()
        else:
            # Missing tag, cannot parse
            return None
    
    # Sort tags by their positions in the string
    sorted_tags = sorted(tag_positions.items(), key=lambda x: x[1])
    
    # Extract content between tags
    for i, (tag, pos) in enumerate(sorted_tags):
        # Find start position (after the tag)
        tag_end = pos + len(f'[{tag}]')
        
        # Find end position (start of next tag or end of string)
        if i < len(sorted_tags) - 1:
            next_tag_pos = sorted_tags[i + 1][1]
            content = triplet_str[tag_end:next_tag_pos]
        else:
            content = triplet_str[tag_end:]
        
        # Clean up content
        content = content.strip()
        
        # Map to correct variable based on tag
        if tag == 'A':
            aspect = content
        elif tag == 'O':
            opinion = content
        elif tag == 'S':
            sentiment = content
    
    # Return tuple if all three components found
    if aspect is not None and opinion is not None and sentiment is not None:
        return (aspect, opinion, sentiment)
    else:
        return None

def parse_multiple_triplets(triplet_str, separator='[SSEP]'):
    """
    Parse multiple AOS triplets separated by a delimiter.
    
    Args:
        triplet_str: String with multiple triplets like "[A] service [O] good [S] positive [SSEP] [A] place [O] nice [S] positive"
        separator: Separator between triplets (default: '[SSEP]')
    
    Returns:
        list: List of (aspect, opinion, sentiment) tuples
    """
    # Split by separator and parse each triplet
    triplets_str = triplet_str.split(separator)
    triplets = []
    
    for t in triplets_str:
        t = t.strip()
        if t == '':
            return [] 
            
        parsed = parse_aos_triplet(t)
        if parsed:
            triplets.append(parsed)
        else:
            print(f"Warning: Could not parse triplet: '{t}'")
    
    return triplets

In [ ]:

def filter_valid_data_for_counterfacts(lang, split='dev_test'):
	"""
	Filter valid test data for counterfactuals based on the language.
	The data have to fulfill the following conditions:
	- Only one triplet per index
	- No 'null' aspect or opinion in the triplet
	- Only the [A] [O] [S] ordering is used
	Args:
		language (str): The language of the dataset to filter.
	Returns:
		dict: A dictionary with filtered data containing 'index', 'original_sentence', and 'original_triplet'.
	"""
	counterfact_data_dict = {
		'index': [],
		'original_sentence': [],
		'original_triplet': []
	}

	# Read clean data
	no_reasoning_suffix = '_noreasoning' if split == 'train' else ''
	dataset_path = f'hotel_dataset/{lang}/clean_train2500/hotel_aste_{split}_augmented{no_reasoning_suffix}.json'
	with open(dataset_path, 'r', encoding='utf-8') as f:
		dataset_json = json.load(f)
	print(f"Dataset loaded from {dataset_path} ({len(dataset_json)} entries)")

	# Get only the [A] [O] [S] ordering
	for i in range(0, len(dataset_json), 5):
		counterfact_data_dict['index'].append(dataset_json[i]['sentence_id'])
		counterfact_data_dict['original_sentence'].append(dataset_json[i]['input'].replace(' [A] [O] [S]', ''))
		counterfact_data_dict['original_triplet'].append(parse_multiple_triplets(dataset_json[i]['target']))

	# Filter out triplets with 'null' aspect or opinion
	# and ensure only one triplet per index
	valid_indexes = []
	for idx, triplets in enumerate(counterfact_data_dict['original_triplet']):
		if len(triplets) == 1:
			triplet = triplets[0]
			aspect, opinion, sentiment = triplet
			if aspect == 'null' or aspect == '':
				# print(f"Skipping index {idx} with triplet: {triplet} (aspect is 'null')")
				continue
			if opinion == 'null' or opinion == '':
				# print(f"Skipping index {idx} with triplet: {triplet} (opinion is 'null')")
				continue
			valid_indexes.append(idx)

	# Create a new dictionary with only valid indexes
	valid_init_counterfacts = {
		'index': [counterfact_data_dict['index'][i] for i in valid_indexes],
		'original_sentence': [counterfact_data_dict['original_sentence'][i] for i in valid_indexes],
		'original_triplet': [str(counterfact_data_dict['original_triplet'][i]) for i in valid_indexes]
	}
	return valid_init_counterfacts

# List all files in a directory recursively, but stop at the last folder before a file
def list_files_recursively(directory):
	file_list = []
	for root, dirs, files in os.walk(directory):
		for file in files:
			file_list.append(os.path.join(root, file))
	return file_list


# Filter all of the counterfactuals for each model
def get_test_data_correct_predictions(lang, valid_init_counterfacts, filtered_data_dir='temp'):
	"""
	Get the test data for correct predictions based on the language.
	Args:
		lang (str): The language of the dataset to filter.
		valid_init_counterfacts (dict): The initial valid counterfactuals data containing 'index', 'original_sentence', and 'original_triplet'.
	Returns:
		dict: A dictionary with filtered data containing 'index', 'original_sentence', and 'original_triplet'.
	"""

	filtered_dfs = {}
	for model_path in models:
		if f"circuit-{lang}" not in model_path:
			print(f"Skipping model {model_path} as it does not match the language {lang}")
			continue
		model = load_finetuned_model_lens_from_dir(model_path)
		device = (
			torch.device("mps") if torch.backends.mps.is_available()
			else torch.device("cuda") if torch.cuda.is_available()
			else torch.device("cpu")
		)
		model.to(device)
		model.eval()

		print(model_path)

		seed = model_path.split('/')[4]
		filtered_data_path = os.path.join(filtered_data_dir, f'{lang}_{seed}.csv')
		df = pd.DataFrame(valid_init_counterfacts)
		os.makedirs(os.path.dirname(filtered_data_path), exist_ok=True)
		id = filtered_data_path.split('/')[-1]
		filtered_df = filter_correct_data(
			model,
			df,
			"original_sentence",
			"original_triplet",
			filter_mode="AOS",
			max_tokens=100,
			filter_only_correct=True,
			save_path=filtered_data_path
		)
		filtered_dfs[id] = filtered_df.copy()
	return filtered_dfs

In [ ]:
valid_init_counterfacts = filter_valid_data_for_counterfacts('sunda', split='dev_test')

In [ ]:
files = list_files_recursively('outputs/models/eap')
models = [os.path.dirname(f) for f in files]
models = list(set(models))
models

In [ ]:
filtered_dfs = get_test_data_correct_predictions('sunda', valid_init_counterfacts, filtered_data_dir='temp/train2500')

In [ ]:
filtered_dfs = {}
results_path = os.listdir('temp/train2500')
for path in results_path:
    filtered_dfs[path] = pd.read_csv(os.path.join('temp/train2500', path))
print(filtered_dfs.keys())
print(len(filtered_dfs))

In [ ]:
indexes = set()
first = True
for df in filtered_dfs.values():
	if first:
		indexes = set(df['index'].tolist())
		first = False
	else:
		# Get the intersection of indexes
		indexes.intersection_update(df['index'].tolist())

# Convert to list and sort
indexes = sorted(list(indexes))
len(indexes)

In [ ]:
# Sort the filtered_dfs by key
filtered_dfs = dict(sorted(filtered_dfs.items()))

In [ ]:
for key, filtered_df in filtered_dfs.items():
    print(f"Model: {key}, Number of filtered instances: {len(filtered_df)}")

In [ ]:
for lang in ['indo', 'eng', 'sunda']:
	counterfact_data_dict = {
		'index': [],
		'original_sentence': [],
		'original_triplet': []
	}

	dataset_train_path = f'hotel_dataset/{lang}/clean_train2500/hotel_aste_train_augmented_noreasoning.json'
	dataset_dev_test_path = f'hotel_dataset/{lang}/clean_train2500/hotel_aste_dev_test_augmented.json'
	with open(dataset_train_path, 'r', encoding='utf-8') as f:
		dataset_train = json.load(f)
	with open(dataset_dev_test_path, 'r', encoding='utf-8') as f:
		dataset_dev_test = json.load(f)

	for i in range(0, len(dataset_dev_test), 5):
		if dataset_dev_test[i]['sentence_id'] not in indexes:
			continue
		counterfact_data_dict['index'].append(dataset_dev_test[i]['sentence_id'])
		counterfact_data_dict['original_sentence'].append(dataset_dev_test[i]['input'])
		counterfact_data_dict['original_triplet'].append(dataset_dev_test[i]['target'])

	df_counterfact = pd.DataFrame(counterfact_data_dict)
	df_counterfact['original_pair'] = df_counterfact.apply(lambda row: f"{row['original_sentence']} {row['original_triplet']}", axis=1)
	df_counterfact['corrupted_pair'] = np.nan

	df_counterfact[['index', 'original_pair', 'corrupted_pair']].to_csv(f'hotel_dataset/empty_counterfacts/{lang}_counterfacts.csv', index=False)

### Upload models to huggingface

In [ ]:
from huggingface_hub import login
login(token=os.getenv('HF_TOKEN'))

In [ ]:
import os
from huggingface_hub import HfApi

# Instantiate the API client
api = HfApi()

# Define your repository details and the parent directory of your circuits
your_repo_id = "absa-research/train2500"  # Replace with your repository ID
parent_directory = "outputs/models/eap"      # Replace with the path to the 'eap' folder

# List the directories you want to upload
directories_to_upload = [
    "circuit-eng_finetune-eng",
    # "circuit-indo_finetune-indo",
    # "circuit-sunda_finetune-sunda",
]

# Loop through and upload each directory
for dir_name in directories_to_upload:
    local_path = os.path.join(parent_directory, dir_name)
    if os.path.isdir(local_path):
        print(f"Uploading {dir_name}...")
        api.upload_folder(
            folder_path=local_path,
            repo_id=your_repo_id,
            repo_type="model",  # or "dataset", "space"
            path_in_repo=dir_name,  # Creates a folder with the same name in the repo
        )
        print(f"Successfully uploaded {dir_name}.")

print("All specified directories have been uploaded.")

### Debug Counterfact Index

In [ ]:
dataset_paths = {
	'indo': 'hotel_dataset/counterfacts/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv',
	'eng': 'hotel_dataset/counterfacts/tflens_hotel-en_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv',
	'sunda': 'hotel_dataset/counterfacts/franken_hotel-su_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
}

dataset_paths_empty = {
    'indo': 'hotel_dataset/empty_counterfacts/indo_counterfacts.csv',
	'eng': 'hotel_dataset/empty_counterfacts/eng_counterfacts.csv',
	'sunda': 'hotel_dataset/empty_counterfacts/sunda_counterfacts.csv'
}

In [ ]:
dfs_counterfact = {
    'indo': pd.read_csv(dataset_paths['indo']),
    'eng': pd.read_csv(dataset_paths['eng']),
    'sunda': pd.read_csv(dataset_paths['sunda'])
}
dfs_empty_counterfacts = {
    'indo': pd.read_csv(dataset_paths_empty['indo']),
    'eng': pd.read_csv(dataset_paths_empty['eng']),
    'sunda': pd.read_csv(dataset_paths_empty['sunda'])
}

In [ ]:
for lang in dfs_empty_counterfacts.keys():
    dfs_empty_counterfacts[lang]['original_sentence'] = dfs_empty_counterfacts[lang]['original_pair'].apply(lambda x: x.split(' [A] [O] [S] ')[0].strip())

In [ ]:
# Check if the original sentences in the empty counterfacts are in the existing counterfacts
def check_empty_counterfacts_in_existing(existing, empty):
	existing_set = set(existing)
	empty_set = set(empty)
	existing_set = {s.strip() for s in existing_set}  # Clean up whitespace
	empty_set = {s.strip() for s in empty_set}  # Clean up whitespace
	intersection = existing_set.intersection(empty_set)
	return intersection

exist_dict = {}
for lang in dfs_empty_counterfacts.keys():
	exist_dict[lang] = check_empty_counterfacts_in_existing(dfs_counterfact[lang]['original_sentence'], dfs_empty_counterfacts[lang]['original_sentence'])
	exist_dict[lang] = list(exist_dict[lang])  # Convert to list for easier handling

In [ ]:
exist_dict_indexes = {}

In [ ]:
# Get index of each instance in the empty counterfacts that exists in the existing counterfacts
for lang in dfs_empty_counterfacts.keys():
	exist_dict_indexes[lang] = []
	for sentence in exist_dict[lang]:
		exist_dict_indexes[lang].extend(dfs_counterfact[lang].loc[sentence == dfs_counterfact[lang]['original_sentence'], :].index.tolist())

In [ ]:
len(exist_dict_indexes['indo']), len(exist_dict_indexes['eng']), len(exist_dict_indexes['sunda'])

In [ ]:
difference = set(exist_dict_indexes['sunda']) - set(exist_dict_indexes['indo'])  # Check if there are any indexes in eng that are not in indo

In [ ]:
difference

In [ ]:
dfs_counterfact['indo'].loc[list(difference), :]

In [ ]:
dfs_counterfact['eng'].loc[list(difference), :]

In [ ]:
dfs_counterfact['sunda'].loc[list(difference), :]

In [ ]:
dfs_empty_counterfacts['indo'].to_csv('test_indo.csv', index=False)